# 02 — Nettoyage et préparation

Déduplication (hash perceptuel), redimensionnement 640×640, augmentation, conversion TACO COCO → YOLO via `scripts/taco_to_yolo.py`.

**Split géographique :** séparer par parcours / zone géographique, jamais de quasi-doublons train/test (split par parcours).

In [ ]:
from pathlib import Path
from PIL import Image, ImageOps
import hashlib

def phash_simple(path: Path, size: int = 16) -> int:
    img = ImageOps.grayscale(Image.open(path).resize((size, size)))
    pixels = list(img.getdata())
    avg = sum(pixels) / len(pixels)
    return int("".join("1" if p > avg else "0" for p in pixels), 2)

def dedupe(folder: Path):
    seen = set()
    for p in sorted(folder.glob("**/*.jpg")):
        key = phash_simple(p)
        if key in seen:
            p.unlink()
            print("supprimé (doublon)", p)
        seen.add(key)

In [ ]:
# Conversion TACO -> YOLO avec mapping vers 5 classes
# python ../scripts/taco_to_yolo.py --taco-dir ../datasets/taco --out-dir ../datasets/taco/yolo_cleanspot

## Augmentation (côté Ultralytics)

Réglée dans `configs/train_detection.yaml` : mosaic, HSV, flips, scale.